# SQL Single-Table Query Strategies

**Purpose:** This guide is a quick-reference strategy sheet for SQL interview questions that involve transforming, analyzing, or reshaping data within a single table. It covers the most common patterns you'll encounter — comparing rows, aggregating, ranking, handling dates, pivoting, and more — with a focus on identifying the right approach before writing any code, choosing the most efficient function, and explaining your reasoning clearly to an interviewer.

<hr style="border: 3px solid black;">

## Table of Contents

1. [First Step: Identify the Pattern](#1-first-step-identify-the-pattern)
2. [Decision Tree (5-Second Version)](#2-decision-tree-5-second-version)
3. [Pattern Library with Examples](#3-pattern-library-with-examples)
    - [A. Row-Level Filtering](#pattern-a-row-level-filtering) — WHERE + ORDER BY
    - [B. Compare Rows](#pattern-b-compare-rows) — LAG / LEAD / Self Join
    - [C. Date Operations](#pattern-c-date-operations) — EXTRACT / DATEDIFF / INTERVAL
    - [D. Combine Rows](#pattern-d-combine-rows) — GROUP BY + CASE / Self Join
    - [E. Ranking](#pattern-e-ranking) — ROW_NUMBER / RANK / DENSE_RANK
    - [F. Running & Rolling Calculations](#pattern-f-running-totals) — SUM() OVER() / ROWS BETWEEN / Rolling Windows
    - [G. Missing Data](#pattern-g-missing-data) — NOT EXISTS / LEFT JOIN + IS NULL
    - [H. Deduplication](#pattern-h-deduplication) — ROW_NUMBER / DISTINCT ON / GROUP BY
    - [I. Conditional Logic](#pattern-i-conditional-logic) — CASE WHEN
    - [J. Filtering After Aggregation](#pattern-j-filtering-after-aggregation) — HAVING / Self Join / CTE + Join
    - [K. Pivoting](#pattern-k-pivoting) — CASE + GROUP BY
    - [L. String & Text Operations](#pattern-l-string-text-operations) — LIKE, Regex, LENGTH, UPPER/LOWER, SUBSTRING, STRING_AGG
    - [M. Conditional Aggregation](#pattern-m-conditional-aggregation) — CASE inside SUM/COUNT/AVG for rates, percentages, conditional sums
4. [Critical Efficiency Rules](#4-critical-efficiency-rules)
5. [GROUP BY vs Window Functions](#5-group-by-vs-window-functions)
6. [Method Selection: How to Know All Your Options](#6-method-selection-framework) — CTE vs Subquery, 3-Method Mental Model, Efficiency Rankings
7. [How to Think from the Vignette](#7-how-to-think-from-the-vignette)
8. [Interview Prompt Templates](#8-interview-prompt-templates)
9. [What to Avoid](#9-what-to-avoid)
10. [Final Takeaway](#10-final-takeaway)

<hr style="border: 3px solid black;">

<a id='1-first-step-identify-the-pattern'></a>

## 1. First Step: Identify the Pattern

Before writing **any** SQL, ask yourself:

> *What is this question actually asking me to do?*

The table below combines **pattern recognition**, **data type cues**, **function selection**, and **efficiency ranking** into a single reference. Functions are listed from most efficient (1) to least efficient (3).

| Pattern | Signal Words | Common Data Types | Rank | Function | Rationale |
|---|---|---|---|---|---|
| **Filter rows** | where, find, report, not equal to, odd/even, greater/less than | Any type | 1 | `WHERE` + operators (`=`, `!=`, `>`, `AND`, `OR`, `MOD`) | Direct row selection; single scan with index use |
| | | | 2 | `CASE` inside `WHERE` | Useful for complex multi-condition logic |
| **Compare rows** | previous, next, yesterday, consecutive, following | Date/timestamp, ordered values | 1 | `LAG()` (look back) / `LEAD()` (look ahead) | Single pass; designed for ordered row comparison |
| | | | 2 | Self `JOIN` | Works but adds extra join overhead |
| | | | 3 | Correlated subquery | Re-executes for every row; slowest |
| **Date operations** | days between, month, year, age, interval, gap | Date/timestamp | 1 | `DATEDIFF()` / `DATE_PART()` / `EXTRACT()` | Built-in date math; single pass |
| | | | 2 | `INTERVAL` arithmetic (e.g., `+ INTERVAL '1 day'`) | Flexible but syntax varies by dialect |
| | | | 3 | Manual cast/subtraction | Error-prone; avoid when date functions exist |
| **Combine rows** | avg, sum, duration, per X | Numeric, categorical (start/end), IDs | 1 | `GROUP BY` (+ `CASE`) | Direct aggregation; no duplicate rows produced |
| | | | 2 | Self `JOIN` | Clean pairing but more work than GROUP BY |
| | | | 3 | Window function | Creates duplicates that need removal |
| **Rank rows** | top, latest, first, nth | Ordered values, date/timestamp | 1 | `ROW_NUMBER()` | Direct ranking with partition control |
| | | | 2 | `RANK()` / `DENSE_RANK()` | Use when ties matter; slightly more complex |
| | | | 3 | Correlated subquery | Runs per row; poor performance on large tables |
| **Running & rolling calc** | cumulative, rolling, moving, 7-day window, trailing N days | Numeric, date/timestamp | 1 | CTE + `SUM/AVG OVER (ROWS BETWEEN)` | Pre-aggregate, then single window pass |
| | | | 2 | CTE + `SUM OVER` + divide by N | Same plan; derive avg from sum |
| | | | 3 | CTE + Self `JOIN` on date range | Repeated matching per day; heavier |
| | | | 4 | CTE + Correlated subqueries | Re-sums per row, twice; slowest |
| **Existence** | no, missing, without, never | IDs, any type | 1 | `NOT EXISTS` | Handles NULLs correctly; stops at first match |
| | | | 2 | `LEFT JOIN` + `IS NULL` | Valid anti-join; slightly more verbose |
| | | | 3 | `NOT IN` | Fails silently when NULLs present in subquery |
| **Deduplicate** | unique, one per, most recent | IDs, date/timestamp | 1 | `ROW_NUMBER()` | Full control over which row to keep |
| | | | 2 | `DISTINCT` | Simple but no control over row selection |
| | | | 3 | Self `JOIN` | Overly complex for deduplication |
| **Conditional logic** | label, categorize, bucket, if/then, flag | Any type | 1 | `CASE WHEN` (standalone) | Row-level labeling; no aggregation needed |
| | | | 2 | `CASE` inside `COUNT` / `SUM` | Count or sum by category in one query |
| | | | 3 | Multiple queries with `WHERE` | Separate query per category; redundant work |
| **Filter groups** | more than, at least, groups where, having | Numeric (aggregated) | 1 | `GROUP BY` + `HAVING` | Filters groups after aggregation; designed for this |
| | | | 2 | Subquery + `WHERE` | Works but adds nesting; less readable |
| **Pivot** | rows to columns, side by side, crosstab | Categorical + numeric | 1 | `CASE` + `GROUP BY` (manual pivot) | Portable; works in all dialects |
| **String & text ops** | like, contains, starts with, email, valid, case, substring, concatenate, group strings | Text, email | 1 | `LIKE` + wildcards, regex (`~`), `LENGTH()`, `UPPER()`/`LOWER()`, `SUBSTRING()`, `STRING_AGG()` | Direct text filtering and transformation; aggregates per group |
| | | | 2 | Manual string parsing / application-level filtering | More complex but portable |
| **Conditional agg** | rate, percentage, approval, confirmed, approved vs total, monthly breakdown | Numeric (amounts), categorical (status) | 1 | `CASE` inside `SUM()` / `COUNT()` / `AVG()` | One pass; computes multiple metrics in single GROUP BY |
| | | | 2 | Multiple queries with WHERE | Separate queries; redundant scans |
| | | | 2 | `PIVOT` keyword (SQL Server / Oracle) | Cleaner syntax but dialect-specific |
| | | | 3 | Application-level pivot | Moves work outside SQL; last resort |

<hr style="border: 3px solid black;">

<a id='2-decision-tree-5-second-version'></a>

## 2. Decision Tree (5-Second Version)

Use this quick mental checklist when you first read a problem — **the first YES is your answer:**

<div class="fc">
  <div class="fc-node fc-start">READ THE QUESTION<br/>What am I being asked to do?</div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">Just filtering rows by conditions on columns?</div>
  <div class="fc-node fc-good"><a href="#pattern-a-row-level-filtering" data-no-arrow>YES → <strong>WHERE + ORDER BY</strong></a><code>SELECT cols FROM table
WHERE condition ORDER BY col</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Does it involve dates?</div>
  <div class="fc-node fc-good"><a href="#pattern-c-date-operations" data-no-arrow>YES → <strong>See DATE FORK below ↓</strong></a></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Am I comparing rows?<br/>(prev vs next, row pairs)</div>
  <div class="fc-node fc-good"><a href="#pattern-b-compare-rows" data-no-arrow>YES → <strong>LAG() / LEAD()</strong></a> — window functions<code>LAG(col)  OVER(ORDER BY date_col) AS prev_val
LEAD(col) OVER(ORDER BY date_col) AS next_val</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Am I collapsing rows into a summary?<br/>(count, sum, avg)</div>
  <div class="fc-node fc-good"><a href="#pattern-d-combine-rows" data-no-arrow>YES → <strong>GROUP BY + aggregate function</strong></a><code>SELECT group_col, COUNT(*), SUM(val), AVG(val)
FROM table GROUP BY group_col</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Filtering AFTER grouping?<br/>("groups where count > 3")</div>
  <div class="fc-node fc-good"><a href="#pattern-j-filtering-after-aggregation" data-no-arrow>YES → <strong>GROUP BY + HAVING</strong></a><code>SELECT group_col, COUNT(*)
FROM table GROUP BY group_col
HAVING COUNT(*) > 3</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Am I ranking rows?<br/>("top N per group")</div>
  <div class="fc-node fc-good"><a href="#pattern-e-ranking" data-no-arrow>YES → <strong>ROW_NUMBER() / RANK()</strong></a> — window functions<code>ROW_NUMBER() OVER(
  PARTITION BY group_col
  ORDER BY val DESC) AS rn</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Running total / cumulative?</div>
  <div class="fc-node fc-good"><a href="#pattern-f-running-totals" data-no-arrow>YES → <strong>SUM() OVER()</strong></a> — window function<code>SUM(amount) OVER(
  ORDER BY date_col) AS running_total</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Labeling / categorizing?<br/>("if X then label Y")</div>
  <div class="fc-node fc-good"><a href="#pattern-i-conditional-logic" data-no-arrow>YES → <strong>CASE WHEN</strong></a><code>CASE WHEN val > 100 THEN 'High'
     WHEN val > 50  THEN 'Medium'
     ELSE 'Low' END AS tier</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Pivoting rows → columns?</div>
  <div class="fc-node fc-good"><a href="#pattern-k-pivoting" data-no-arrow>YES → <strong>CASE + GROUP BY (pivot)</strong></a><code>SELECT grp,
  SUM(CASE WHEN cat='A' THEN val ELSE 0 END) AS A,
  SUM(CASE WHEN cat='B' THEN val ELSE 0 END) AS B
FROM table GROUP BY grp</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Finding missing data?</div>
  <div class="fc-node fc-good"><a href="#pattern-g-missing-data" data-no-arrow>YES → <strong>NOT EXISTS / IS NULL</strong></a><code>WHERE NOT EXISTS (
  SELECT 1 FROM t2 WHERE t2.id = t1.related_id)</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-good"><a href="#pattern-h-deduplication" data-no-arrow>Still here? → <strong>Removing duplicates</strong></a><code>ROW_NUMBER() OVER(
  PARTITION BY key_col
  ORDER BY date DESC) = 1</code></div>
</div>


<hr style="border: 3px solid black;">

<a id='3-pattern-library-with-examples'></a>

## 3. Pattern Library with Examples

<a id='pattern-a-row-level-filtering'></a>

### A. Row-Level Filtering (WHERE + ORDER BY)

**Signals:** "find rows where," "report all X that satisfy," "not equal to," "odd-numbered," "greater than," "exclude," "not boring"

---

**Best approach — `WHERE` + operators**

**Method:** Apply conditions directly on existing columns using `WHERE`. Combine multiple conditions with `AND` / `OR`. Use `ORDER BY` when the output needs sorting. No aggregation, no window functions — just selecting the rows that match.

**In plain language:** "Look at each row, check if it meets the criteria, and return the ones that do."

```sql
-- Find movies with odd-numbered ID and description not 'boring', sorted by rating
SELECT *
FROM Cinema
WHERE id % 2 = 1
  AND description != 'boring'
ORDER BY rating DESC;
```

---

**Common WHERE Operators**

| Operator | Purpose | Example |
|---|---|---|
| `=`, `!=` (or `<>`) | Exact match / exclusion | `description != 'boring'` |
| `>`, `<`, `>=`, `<=` | Numeric / date comparison | `rating >= 8.0` |
| `%` (MOD) | Odd/even, divisibility | `id % 2 = 1` (odd) |
| `AND`, `OR` | Combine conditions | `WHERE a > 5 AND b = 'x'` |
| `IN (...)` | Match any in a list | `status IN ('active', 'pending')` |
| `NOT IN (...)` | Exclude a list | `category NOT IN ('spam', 'test')` |
| `BETWEEN` | Range (inclusive) | `rating BETWEEN 7.0 AND 9.0` |
| `LIKE` / `NOT LIKE` | Pattern matching | `name LIKE 'A%'` (starts with A) |
| `IS NULL` / `IS NOT NULL` | NULL checks | `email IS NOT NULL` |

---

**How to recognize this pattern:**

The question gives you conditions on **existing columns** and asks you to return matching rows — no aggregation, no comparison between rows, no grouping. The output shape is the **same columns as the input** (or a subset), just fewer rows.

> **Interview tip:** These problems look trivially easy, but interviewers use them to check if you write clean, correct SQL under pressure. Pay attention to edge cases like NULLs in `!=` comparisons (NULL != 'boring' evaluates to NULL, not TRUE) and whether `MOD` syntax varies by dialect (`%` in MySQL/PostgreSQL, `MOD()` function in Oracle).

<a id='pattern-b-compare-rows'></a>

### B. Compare Rows

**Signals:** previous, yesterday, next, consecutive, following

---

**Best approach — Window Function: `LAG()` (look backward)**

**Method:** `LAG()` is a window function that lets you access the previous row's value without a join. You define the order with `OVER (ORDER BY ...)` and it hands you the value from the row right before the current one. Use `LAG()` when the question asks about what came before — "previous day," "yesterday," "prior month."

**In plain language:** The inner query adds two new columns to every row — the previous day's date and the previous day's temperature. Then the outer query simply checks: "Is today exactly one day after yesterday, and is today's temperature higher?" If both are true, we keep that row's ID.

```sql
SELECT id
FROM (
    SELECT
        id,
        recordDate,
        temperature,
        LAG(recordDate) OVER (ORDER BY recordDate) AS prev_date,
        LAG(temperature) OVER (ORDER BY recordDate) AS prev_temp
    FROM Weather
) t
WHERE recordDate = prev_date + INTERVAL '1 day'
  AND temperature > prev_temp;
```

---

**Also common — Window Function: `LEAD()` (look forward)**

**Method:** `LEAD()` is the mirror image of `LAG()` — instead of looking at the previous row, it looks at the next row. Use `LEAD()` when the question asks about what comes after — "next purchase," "following day," "will the customer return."

**In plain language:** For every row, we peek ahead at the next row's value. This is useful when you need to ask "what happens next?" For example, finding users whose next login was more than 30 days later — that gap tells you they churned.

```sql
SELECT
    user_id,
    login_date,
    LEAD(login_date) OVER (PARTITION BY user_id ORDER BY login_date) AS next_login,
    LEAD(login_date) OVER (PARTITION BY user_id ORDER BY login_date) - login_date AS days_until_next
FROM Logins;
```

---

**When to use LAG vs LEAD**

| Question asks about... | Use | Example |
|---|---|---|
| What came **before** this row | `LAG()` | "Was yesterday's temperature lower?" |
| What comes **after** this row | `LEAD()` | "When is the user's next login?" |
| The **gap** between consecutive rows | Either works | "How many days between orders?" |

**In plain language:** Think of it like standing in a line. `LAG()` lets you turn around and look at the person behind you. `LEAD()` lets you look at the person in front of you. Both are window functions, both use `OVER (ORDER BY ...)`, and the syntax is identical — the only difference is the direction.

---

**Alternative — Self Join**

**Method:** Join the table to itself by matching each row to the row from the day before. No window function needed, but the database has to do extra work to pair up the rows.

**In plain language:** We take two copies of the same Weather table and line them up so that each day in the first copy is matched to the day before it in the second copy. Then we just check which days were warmer than the day before.

```sql
SELECT w1.id
FROM Weather w1
JOIN Weather w2
  ON w1.recordDate = w2.recordDate + INTERVAL '1 day'
WHERE w1.temperature > w2.temperature;
```

<hr style="border: 2px solid black;">

<a id='pattern-c-date-operations'></a>

### C. Date Operations

**Signals:** days between, month, year, age, interval, gap, extract

Date columns show up constantly in single-table interview problems. These functions are often **combined** with other patterns (LAG, GROUP BY, ROW_NUMBER) rather than used alone.

---

**Extracting parts of a date — `EXTRACT()` / `DATE_PART()` / `YEAR()`, `MONTH()`, `DAY()`**

**Method:** These functions pull a specific component (year, month, day, hour) out of a date or timestamp column. The syntax varies by dialect but the idea is the same everywhere.

**In plain language:** If you have a column with full dates like `2024-03-15` and the question asks "how many orders per month," you need to break that date into its month component first, then group by it. Think of it like opening a date and pulling out just the piece you need.

```sql
-- PostgreSQL / standard SQL
SELECT EXTRACT(MONTH FROM order_date) AS order_month, COUNT(*) AS total_orders
FROM Orders
GROUP BY EXTRACT(MONTH FROM order_date);

-- MySQL
SELECT MONTH(order_date) AS order_month, COUNT(*) AS total_orders
FROM Orders
GROUP BY MONTH(order_date);

-- SQL Server
SELECT DATEPART(MONTH, order_date) AS order_month, COUNT(*) AS total_orders
FROM Orders
GROUP BY DATEPART(MONTH, order_date);
```

---

**Calculating differences between dates — `DATEDIFF()` / subtraction / `AGE()`**

**Method:** These calculate the gap between two dates. Often paired with `LAG()` or `LEAD()` to find the time between consecutive rows in the same table. Syntax varies heavily by dialect.

**In plain language:** When you need to answer "how many days between event A and event B," you need date difference. If both events live in the same row (like a start and end column), you just subtract. If they live in different rows (like consecutive logins), you first use `LAG()` or `LEAD()` to bring them onto the same row, then subtract.

```sql
-- PostgreSQL: subtract dates directly (returns integer days)
SELECT order_date - LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS days_since_last
FROM Orders;

-- MySQL: use DATEDIFF
SELECT DATEDIFF(order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date)) AS days_since_last
FROM Orders;

-- SQL Server: use DATEDIFF with unit
SELECT DATEDIFF(DAY, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date), order_date) AS days_since_last
FROM Orders;
```

---

**Date arithmetic — `INTERVAL` / `DATE_ADD()` / `DATEADD()`**

**Method:** Add or subtract a specific amount of time from a date. Used when the question says things like "within 7 days" or "one day after."

**In plain language:** This is for when you need to shift a date forward or backward by a fixed amount — like checking if a row's date is exactly one day after another, or finding all records from the last 30 days.

```sql
-- PostgreSQL
WHERE recordDate = prev_date + INTERVAL '1 day'

-- MySQL
WHERE recordDate = DATE_ADD(prev_date, INTERVAL 1 DAY)

-- SQL Server
WHERE recordDate = DATEADD(DAY, 1, prev_date)
```

---

**Common interview pattern: dates + LAG/LEAD together**

Most date questions in interviews aren't purely about date math — they combine dates with row comparison. The date functions handle the "how far apart" part, and LAG/LEAD handle the "bring two rows together" part.

| Interview question sounds like... | Approach |
|---|---|
| "Days between consecutive logins" | `LAG()` + date subtraction |
| "Orders per month" | `EXTRACT(MONTH ...)` + `GROUP BY` |
| "Users inactive for 30+ days" | `LEAD()` + `DATEDIFF()` |
| "Was the previous day exactly yesterday?" | `LAG()` + `INTERVAL '1 day'` |
| "First order each year" | `EXTRACT(YEAR ...)` + `ROW_NUMBER()` |

<hr style="border: 2px solid black;">

<a id='pattern-d-combine-rows'></a>

### D. Combine Rows

**Signals:** average, duration, per X, total per group

---

**Best approach — Aggregate Function: `GROUP BY` + Conditional Aggregation with `CASE`**

**Method:** `GROUP BY` collapses multiple rows into one row per group. Inside the aggregation, `CASE` expressions act like an if/then switch — they pick out specific values (like the "start" timestamp vs the "end" timestamp) so you can combine them in one pass.

**In plain language:** The inner query groups every activity by machine and process, then uses `CASE` to grab the end timestamp and the start timestamp separately, and subtracts them to get the process time. The outer query then takes all those process times for each machine and averages them. Two levels of grouping, but the logic reads top to bottom: first get each process duration, then average them per machine.

```sql
SELECT
    machine_id,
    ROUND(AVG(process_time)::numeric, 3)
FROM (
    SELECT
        machine_id,
        process_id,
        MAX(CASE WHEN activity_type = 'end' THEN timestamp END) -
        MAX(CASE WHEN activity_type = 'start' THEN timestamp END) AS process_time
    FROM Activity
    GROUP BY machine_id, process_id
) t
GROUP BY machine_id;
```

---

**Alternative — Self JOIN (pair start and end rows directly)**

**Method:** Join the table to itself, matching start rows to their corresponding end rows. Then aggregate the differences. This avoids the CASE expression by explicitly pairing the rows.

**In plain language:** Take two copies of the Activity table — one filtered to "start" events, one to "end" events — and match them on machine and process. Subtract to get duration, then average.

```sql
SELECT
    s.machine_id,
    ROUND(AVG(e.timestamp - s.timestamp)::numeric, 3) AS processing_time
FROM Activity s
JOIN Activity e
    ON s.machine_id = e.machine_id
    AND s.process_id = e.process_id
    AND s.activity_type = 'start'
    AND e.activity_type = 'end'
GROUP BY s.machine_id;
```

---

**Efficiency ranking for this pattern:**

| Rank | Method | Why |
|---|---|---|
| 1 | GROUP BY + CASE (conditional aggregation) | Single scan, no join overhead |
| 2 | Self JOIN + GROUP BY | Clean and readable, but join has a cost |
| 3 | Correlated subquery | Runs per row — avoid |

<hr style="border: 2px solid black;">

<a id='pattern-e-ranking'></a>

### E. Ranking

**Signals:** top, latest, first, nth

---

**Best approach — Window Function: `ROW_NUMBER()`**

**Method:** `ROW_NUMBER()` is a window function that assigns a sequential number to each row within a partition. `PARTITION BY` splits the data into groups (like one group per customer), and `ORDER BY` controls which row gets number 1 within each group.

**In plain language:** The inner subquery is adding a row number to every row for a given customer_id, starting with the latest date as number 1, the second latest as number 2, and so on. So when we get to the outer query and filter for `rn = 1`, we are keeping only the most recent order for each customer and throwing away all the older ones.

```sql
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY order_date DESC
           ) AS rn
    FROM Orders
) t
WHERE rn = 1;
```

---

**When to use `RANK()` or `DENSE_RANK()` instead**

**Method:** `RANK()` and `DENSE_RANK()` are also window functions, but they handle ties differently. `RANK()` skips numbers after a tie (1, 1, 3), while `DENSE_RANK()` does not skip (1, 1, 2). Use these when the problem says "top N" and ties should be included.

**In plain language:** If two customers placed orders on the exact same date and you want to keep both of them (not randomly pick one), use `RANK()` instead of `ROW_NUMBER()`. The rest of the query stays the same — just swap the function name.

<hr style="border: 2px solid black;">

<a id='pattern-f-running-totals'></a>

### F. Running & Rolling Calculations

**Signals:** cumulative, rolling, moving average, running sum, year-to-date, 7-day window, trailing N days

This pattern covers two related use cases: **unbounded running totals** (sum everything from the start up to the current row) and **bounded rolling windows** (sum or average over a fixed trailing period like 7 days). The key difference is the window frame clause.

---

**Unbounded Running Total — `SUM() OVER(ORDER BY ...)`**

**Method:** With no explicit frame clause, `SUM() OVER(ORDER BY ...)` defaults to summing from the first row up to the current row. Each row's total grows because it includes everything before it.

**In plain language:** "For every row, add up everything from the beginning of the table to here."

```sql
SELECT
    sale_date,
    amount,
    SUM(amount) OVER (ORDER BY sale_date) AS running_total
FROM Sales;
```

---

**Bounded Rolling Window — `ROWS BETWEEN N PRECEDING AND CURRENT ROW`**

**Method:** Adding a frame clause like `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` limits the window to a fixed number of rows. Instead of summing from the start, SQL only looks at the current row plus the N rows immediately before it.

**In plain language:** "For every row, only add up the last 7 rows (this one plus the 6 before it) — like a sliding window moving down the table."

```sql
SELECT
    sale_date,
    daily_amount,
    SUM(daily_amount) OVER (
        ORDER BY sale_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_sum,
    ROUND(
        AVG(daily_amount) OVER (
            ORDER BY sale_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS rolling_7day_avg
FROM daily_sales;
```

---

**ROWS vs RANGE — Know the Difference**

| Frame Type | What It Counts | Use When |
|---|---|---|
| `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` | Exactly 6 prior **rows** + current row | Each date has exactly 1 row (pre-aggregated) |
| `RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW` | All rows within the **date range** | Dates may be missing or duplicated |

**When to use which:** If your data has one row per date (because you pre-aggregated), `ROWS` is simpler and faster. If dates can be missing (gaps in the calendar), `RANGE` with an interval handles it correctly because it looks at actual date values, not row positions. In most interview problems, you'll pre-aggregate first and use `ROWS`.

---

**The Pre-Aggregation Step — Why It Matters**

Many rolling window problems have **multiple rows per date** (e.g., multiple customers visiting on the same day). If you apply the window function directly, `ROWS BETWEEN 6 PRECEDING` counts 6 prior *rows*, not 6 prior *days* — giving you the wrong window.

**The fix:** Always aggregate to one row per date first in a CTE, then apply the window.

```sql
-- Step 1: Collapse to one row per day
WITH daily AS (
    SELECT
        visited_on,
        SUM(amount) AS daily_amount
    FROM Customer
    GROUP BY visited_on
)
-- Step 2: Apply rolling window on the clean daily data
SELECT
    visited_on,
    SUM(daily_amount) OVER w AS amount,
    ROUND(AVG(daily_amount) OVER w, 2) AS average_amount
FROM daily
WINDOW w AS (ORDER BY visited_on ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)
ORDER BY visited_on
OFFSET 6;
```

> **Why OFFSET 6?** The first 6 rows don't have a full 7-day window behind them. `OFFSET 6` skips those incomplete windows so the output only shows rows with a complete 7-day average.

> **Why use `WINDOW w AS (...)`?** When you use the same window frame in multiple functions (`SUM` and `AVG`), you can define it once with a named window to keep the query DRY. This is optional — writing the frame inline in each function works the same way.

---

### All Approaches — Ranked by Efficiency

**Approach 1 (Most efficient) — CTE + Window Functions**

Pre-aggregate daily totals, then use `SUM() OVER` and `AVG() OVER` with `ROWS BETWEEN` in a single pass.

```sql
WITH daily AS (
    SELECT visited_on, SUM(amount) AS daily_amount
    FROM Customer
    GROUP BY visited_on
)
SELECT
    visited_on,
    SUM(daily_amount) OVER (
        ORDER BY visited_on
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS amount,
    ROUND(
        AVG(daily_amount) OVER (
            ORDER BY visited_on
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 2
    ) AS average_amount
FROM daily
ORDER BY visited_on
OFFSET 6;
```

**Why it's best:** One aggregation pass, then one window scan. The database computes both the sum and average in a single pass over the sorted daily data.

---

**Approach 2 (Very good) — CTE + Window Sum, then derive average**

Same structure, but compute the rolling sum first, then divide by 7 to get the average. Avoids a second window function call.

```sql
WITH daily AS (
    SELECT visited_on, SUM(amount) AS daily_amount
    FROM Customer
    GROUP BY visited_on
),
rolling AS (
    SELECT
        visited_on,
        SUM(daily_amount) OVER (
            ORDER BY visited_on
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS amount
    FROM daily
)
SELECT
    visited_on,
    amount,
    ROUND(amount / 7.0, 2) AS average_amount
FROM rolling
ORDER BY visited_on
OFFSET 6;
```

**Why it's good:** Functionally identical to Approach 1 — the optimizer may even produce the same plan. Uses one window function instead of two, then derives the average with simple division.

---

**Approach 3 (Less efficient) — Self JOIN on date range**

For each day, join back to all days within the 7-day window, then aggregate.

```sql
WITH daily AS (
    SELECT visited_on, SUM(amount) AS daily_amount
    FROM Customer
    GROUP BY visited_on
)
SELECT
    d1.visited_on,
    SUM(d2.daily_amount) AS amount,
    ROUND(SUM(d2.daily_amount) / 7.0, 2) AS average_amount
FROM daily d1
JOIN daily d2
    ON d2.visited_on BETWEEN d1.visited_on - INTERVAL '6 days'
                          AND d1.visited_on
GROUP BY d1.visited_on
HAVING d1.visited_on >= (
    SELECT MIN(visited_on) + INTERVAL '6 days' FROM daily
)
ORDER BY d1.visited_on;
```

**Why it's worse:** Each day joins to up to 7 other days. On N days, that's up to 7×N row comparisons — compared to a single scan with window functions.

---

**Approach 4 (Least efficient) — Correlated Subqueries**

For each day, run a separate subquery to compute the 7-day sum. The sum is calculated twice (once for the amount column, once for the average).

```sql
WITH daily AS (
    SELECT visited_on, SUM(amount) AS daily_amount
    FROM Customer
    GROUP BY visited_on
)
SELECT
    d.visited_on,
    (SELECT SUM(d2.daily_amount)
     FROM daily d2
     WHERE d2.visited_on BETWEEN d.visited_on - INTERVAL '6 days'
                              AND d.visited_on
    ) AS amount,
    ROUND(
        (SELECT SUM(d2.daily_amount)
         FROM daily d2
         WHERE d2.visited_on BETWEEN d.visited_on - INTERVAL '6 days'
                                  AND d.visited_on
        ) / 7.0, 2
    ) AS average_amount
FROM daily d
WHERE d.visited_on >= (
    SELECT MIN(visited_on) + INTERVAL '6 days' FROM daily
)
ORDER BY d.visited_on;
```

**Why it's worst:** The correlated subquery runs the 7-day sum once per row, and it runs it *twice* (once for `amount`, once for `average_amount`). On N days, that's 2×N separate scans of the daily table.

---

**Efficiency Ranking Summary**

| Rank | Method | Scans | Why |
|---|---|---|---|
| 1 | CTE + `SUM/AVG OVER (ROWS BETWEEN)` | 1 aggregation + 1 window pass | Both calculations in a single scan |
| 2 | CTE + `SUM OVER` + divide by N | 1 aggregation + 1 window pass | Essentially the same; one fewer window call |
| 3 | CTE + Self `JOIN` + `GROUP BY` | 1 aggregation + N×7 join matches | Repeated matching for each day's window |
| 4 | CTE + Correlated subqueries | 1 aggregation + 2×N subquery scans | Re-sums for every row, twice |

> **Interview tip:** Always mention the pre-aggregation CTE step when you see multiple rows per date. Then say: "I'll use `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` for the rolling window since the CTE guarantees one row per day." This shows you understand both the window frame and the data shape.

---

### Worked Example: Rolling 7-Day Restaurant Revenue

This walkthrough shows exactly what happens at every step — how the window builds up, why the first 6 rows are partial, and what `OFFSET 6` does.

---

**Step 1 — Pre-aggregate to daily totals (CTE)**

Multiple customers can visit on the same day. The CTE collapses them into one row per date.

```
+------------+--------------+
| visited_on | daily_amount |
+------------+--------------+
| 2019-01-01 | 100          |
| 2019-01-02 | 110          |
| 2019-01-03 | 120          |
| 2019-01-04 | 130          |
| 2019-01-05 | 110          |
| 2019-01-06 | 140          |
| 2019-01-07 | 150          |
| 2019-01-08 | 80           |
| 2019-01-09 | 110          |
| 2019-01-10 | 280          |
+------------+--------------+
```

---

**Step 2 — Window function processes every row (before OFFSET)**

`ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` means: take the current row plus up to 6 earlier rows in date order. Here's what happens row by row:

```
Row 1: 2019-01-01
  Window = [100]
  rolling_7_day_amount = 100
  rolling_7_day_avg    = 100 / 1 = 100.00      ← only 1 day available

Row 2: 2019-01-02
  Window = [100, 110]
  rolling_7_day_amount = 210
  rolling_7_day_avg    = 210 / 2 = 105.00      ← only 2 days available

Row 3: 2019-01-03
  Window = [100, 110, 120]
  rolling_7_day_amount = 330
  rolling_7_day_avg    = 330 / 3 = 110.00      ← only 3 days available

Row 4: 2019-01-04
  Window = [100, 110, 120, 130]
  rolling_7_day_amount = 460
  rolling_7_day_avg    = 460 / 4 = 115.00      ← only 4 days available

Row 5: 2019-01-05
  Window = [100, 110, 120, 130, 110]
  rolling_7_day_amount = 570
  rolling_7_day_avg    = 570 / 5 = 114.00      ← only 5 days available

Row 6: 2019-01-06
  Window = [100, 110, 120, 130, 110, 140]
  rolling_7_day_amount = 710
  rolling_7_day_avg    = 710 / 6 = 118.33      ← only 6 days available

Row 7: 2019-01-07                               ← FIRST FULL WINDOW
  Window = [100, 110, 120, 130, 110, 140, 150]
  rolling_7_day_amount = 860
  rolling_7_day_avg    = 860 / 7 = 122.86

Row 8: 2019-01-08
  Window = [110, 120, 130, 110, 140, 150, 80]   ← 01-01 slides out
  rolling_7_day_amount = 840
  rolling_7_day_avg    = 840 / 7 = 120.00

Row 9: 2019-01-09
  Window = [120, 130, 110, 140, 150, 80, 110]   ← 01-02 slides out
  rolling_7_day_amount = 840
  rolling_7_day_avg    = 840 / 7 = 120.00

Row 10: 2019-01-10
  Window = [130, 110, 140, 150, 80, 110, 280]   ← 01-03 slides out
  rolling_7_day_amount = 1000
  rolling_7_day_avg    = 1000 / 7 = 142.86
```

---

**Step 3 — Full intermediate result (before OFFSET)**

This is what PostgreSQL computes internally — all 10 rows with their rolling calculations:

```
+------------+--------------+----------------------+-------------------+
| visited_on | daily_amount | rolling_7_day_amount | rolling_7_day_avg |
+------------+--------------+----------------------+-------------------+
| 2019-01-01 | 100          | 100                  | 100.00            |  ← partial (1 day)
| 2019-01-02 | 110          | 210                  | 105.00            |  ← partial (2 days)
| 2019-01-03 | 120          | 330                  | 110.00            |  ← partial (3 days)
| 2019-01-04 | 130          | 460                  | 115.00            |  ← partial (4 days)
| 2019-01-05 | 110          | 570                  | 114.00            |  ← partial (5 days)
| 2019-01-06 | 140          | 710                  | 118.33            |  ← partial (6 days)
| 2019-01-07 | 150          | 860                  | 122.86            |  ✓ full 7-day window
| 2019-01-08 | 80           | 840                  | 120.00            |  ✓ full 7-day window
| 2019-01-09 | 110          | 840                  | 120.00            |  ✓ full 7-day window
| 2019-01-10 | 280          | 1000                 | 142.86            |  ✓ full 7-day window
+------------+--------------+----------------------+-------------------+
```

The first 6 rows have **partial windows** — fewer than 7 days of data. Their averages are mathematically correct for the data available, but they don't represent a true 7-day rolling average.

---

**Step 4 — After OFFSET 6: final output**

`OFFSET 6` tells PostgreSQL to skip the first 6 rows. Nothing is recalculated — it simply hides the partial-window rows:

```
+------------+--------+----------------+
| visited_on | amount | average_amount |
+------------+--------+----------------+
| 2019-01-07 | 860    | 122.86         |
| 2019-01-08 | 840    | 120.00         |
| 2019-01-09 | 840    | 120.00         |
| 2019-01-10 | 1000   | 142.86         |
+------------+--------+----------------+
```

---

**Verification — confirming the final rows by hand:**

```
2019-01-07: [100 + 110 + 120 + 130 + 110 + 140 + 150] = 860  →  860 / 7 = 122.86
2019-01-08: [110 + 120 + 130 + 110 + 140 + 150 +  80] = 840  →  840 / 7 = 120.00
2019-01-09: [120 + 130 + 110 + 140 + 150 +  80 + 110] = 840  →  840 / 7 = 120.00
2019-01-10: [130 + 110 + 140 + 150 +  80 + 110 + 280] = 1000 → 1000 / 7 = 142.86
```

> **Key takeaway:** The window function calculates *all* rows first, then `OFFSET 6` removes the incomplete ones. PostgreSQL doesn't know which rows are "partial" — you do, because you know the window needs 7 days to be meaningful. That's why `OFFSET 6` (not `OFFSET 7`) is correct: the 7th row is the first with a complete window.

<hr style="border: 2px solid black;">

<a id='pattern-g-missing-data'></a>

### G. Missing Data

**Signals:** no, missing, without, never, doesn't exist

---

**Best approach — Subquery Filter: `NOT EXISTS`**

**Method:** `NOT EXISTS` is a subquery filter (not a window function). It checks whether a matching row exists in another table. If no match is found, the row from the outer query is kept. It stops searching as soon as it finds the first match, making it efficient.

**In plain language:** We start with every customer in the Customers table. For each one, we peek into the Orders table and ask "does this customer have any orders?" If the answer is no (NOT EXISTS), we keep that customer. The subquery short-circuits — it stops as soon as it finds even one order.

```sql
SELECT customer_id, customer_name
FROM Customers c
WHERE NOT EXISTS (
    SELECT 1
    FROM Orders o
    WHERE o.customer_id = c.customer_id
);
```

---

**Also strong — `LEFT JOIN` + `WHERE IS NULL`**

**Method:** Join the table to the reference table with a LEFT JOIN, then filter where the joined column is NULL. This identifies rows with no match.

**In plain language:** "Attach orders to each customer. Customers with no orders will have NULL in the order columns. Keep only those."

```sql
SELECT c.customer_id, c.customer_name
FROM Customers c
LEFT JOIN Orders o
    ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL;
```

---

**Avoid — `NOT IN` (with NULLs)**

**Method:** Compare against a list from a subquery. Dangerous if the subquery can return NULLs — the entire NOT IN silently returns zero rows.

**In plain language:** "Check if the ID is NOT IN the list of IDs from Orders." Sounds simple, but if any order has a NULL customer_id, the whole thing breaks and returns nothing.

```sql
-- DANGEROUS if customer_id in Orders can be NULL
SELECT customer_id, customer_name
FROM Customers
WHERE customer_id NOT IN (SELECT customer_id FROM Orders);
```

---

**Efficiency ranking for this pattern:**

| Rank | Method | Why |
|---|---|---|
| 1 | `NOT EXISTS` | Short-circuits on first match, NULL-safe |
| 2 | `LEFT JOIN` + `WHERE IS NULL` | Also efficient, NULL-safe, widely used |
| 3 | `NOT IN` | Breaks with NULLs — avoid unless column is guaranteed NOT NULL |

<hr style="border: 2px solid black;">

<a id='pattern-h-deduplication'></a>

### H. Deduplication

**Signals:** unique, one per, most recent per, latest per group, remove duplicates keeping one

---

**Best approach — Window Function: `ROW_NUMBER()`**

**Method:** Same window function as ranking — `ROW_NUMBER()` with `PARTITION BY` and `ORDER BY`. The difference is intent: here we are not ranking for display, we are numbering rows so we can throw away the duplicates and keep only the one we want (usually the most recent).

**In plain language:** The inner subquery looks at all login records for each user and numbers them starting from the most recent. The outer query keeps only row number 1 — the latest login per user.

```sql
SELECT user_id, login_date, device
FROM (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date DESC) AS rn
    FROM Logins
) t
WHERE rn = 1;
```

---

**Alternative — `GROUP BY` + aggregate (when you only need one column)**

**Method:** If you just need the max or min value per group (not the full row), GROUP BY with MAX/MIN is simpler and avoids the window function overhead.

**In plain language:** "Give me the most recent login date per user." No need to number rows — just take the MAX.

```sql
SELECT user_id, MAX(login_date) AS latest_login
FROM Logins
GROUP BY user_id;
```

---

**Alternative — `DISTINCT ON` (PostgreSQL only)**

**Method:** PostgreSQL's `DISTINCT ON` keeps the first row per group based on an ORDER BY. It's a shortcut for the ROW_NUMBER pattern.

**In plain language:** "For each user, keep only the first row when sorted by login date descending."

```sql
-- PostgreSQL only
SELECT DISTINCT ON (user_id) user_id, login_date, device
FROM Logins
ORDER BY user_id, login_date DESC;
```

---

**Efficiency ranking for this pattern:**

| Rank | Method | Why |
|---|---|---|
| 1 | `DISTINCT ON` (PostgreSQL) | Optimized shortcut, single pass |
| 2 | `ROW_NUMBER()` + filter | Works everywhere, single scan |
| 3 | `GROUP BY` + `MAX/MIN` | Only works when you need the aggregate value, not the full row |
| 4 | Correlated subquery | Runs per row — avoid for dedup |

<hr style="border: 2px solid black;">

<a id='pattern-i-conditional-logic'></a>

### I. Conditional Logic (CASE Expressions)

**Signals:** label, categorize, bucket, if/then, classify, flag, convert rows to columns

---

**Standalone CASE — Categorizing or Bucketing Values**

**Method:** `CASE WHEN ... THEN ... ELSE ... END` is SQL's if/then/else. It evaluates conditions row by row and returns a value based on the first match. Use it standalone in SELECT to create new label columns, or inside WHERE/ORDER BY for conditional filtering and sorting.

**In plain language:** You're looking at each row and sticking a label on it. If the score is above 90, call it "high." If it's between 50 and 90, call it "medium." Everything else is "low." The table stays the same size — you're just adding a new column with your labels.

```sql
SELECT
    user_id,
    score,
    CASE
        WHEN score >= 90 THEN 'high'
        WHEN score >= 50 THEN 'medium'
        ELSE 'low'
    END AS score_tier
FROM Users;
```

---

**CASE inside COUNT / SUM — Counting by Category**

**Method:** Wrapping `CASE` inside an aggregate function like `COUNT()` or `SUM()` lets you count or sum only the rows that match a condition. This is a very common interview pattern for getting counts of different categories in a single query.

**In plain language:** Instead of running three separate queries to count how many users are "high," "medium," and "low," you do it all at once. For each row, `CASE` asks "does this match?" — if yes, it returns 1 (which gets counted), if no, it returns NULL (which gets skipped).

```sql
SELECT
    department,
    COUNT(CASE WHEN status = 'active' THEN 1 END) AS active_count,
    COUNT(CASE WHEN status = 'inactive' THEN 1 END) AS inactive_count,
    COUNT(*) AS total_count
FROM Employees
GROUP BY department;
```

<hr style="border: 2px solid black;">

<a id='pattern-j-filtering-after-aggregation'></a>

### J. Filtering After Aggregation (HAVING)

**Signals:** groups with more than, only customers who bought at least, categories where the average exceeds, managers with at least N reports

---

**Best approach — `GROUP BY` + `HAVING` (direct aggregation)**

**Method:** `HAVING` filters rows *after* `GROUP BY` has collapsed them. `WHERE` filters individual rows before grouping; `HAVING` filters the groups themselves based on aggregate values. You cannot use `WHERE` to filter on `COUNT()`, `SUM()`, `AVG()`, etc. — that's what `HAVING` is for.

**In plain language:** First, GROUP BY creates one row per group. Then HAVING looks at each group and asks "does this group meet my condition?" For example, "only keep customers who have more than 3 orders." WHERE can't do this because at the time WHERE runs, the rows haven't been grouped yet — it doesn't know what the count is.

```sql
-- Customers with more than 3 orders
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM Orders
GROUP BY customer_id
HAVING COUNT(*) > 3;
```

---

**Also strong — Self JOIN + GROUP BY + HAVING**

**Method:** When the output needs columns that aren't in the GROUP BY (like a name from a different row), join the table to itself first, then aggregate. This avoids a separate subquery step.

**In plain language:** Treat the table as two copies — one for the "parent" role and one for the "child" role. Join them, then count and filter. This reads like English: "find managers, count their reports, keep those with 5+."

```sql
-- Managers with at least 5 direct reports (single Employee table)
SELECT m.name
FROM Employee m
JOIN Employee r
    ON m.id = r.managerId
GROUP BY m.id, m.name
HAVING COUNT(*) >= 5;
```

---

**Also valid — Aggregate in CTE/Subquery, then JOIN back**

**Method:** First aggregate and filter in a CTE or subquery, then join the result back to the original table to pick up additional columns (like names). This separates the "find qualifying IDs" step from the "get their details" step.

**In plain language:** Step 1: count reports per manager and keep only those with 5+. Step 2: use those manager IDs to look up names. Two clean steps.

```sql
-- CTE version
WITH qualifying_managers AS (
    SELECT managerId AS id
    FROM Employee
    WHERE managerId IS NOT NULL
    GROUP BY managerId
    HAVING COUNT(*) >= 5
)
SELECT e.name
FROM qualifying_managers AS qm
JOIN Employee AS e
    ON qm.id = e.id;
```

---

**Avoid — Correlated Subquery**

**Method:** For each row, run a separate COUNT query to check if that row qualifies. This works but runs the inner query once per row in the outer table.

**In plain language:** "For every single employee, go count how many people report to them." On a table with 10,000 employees, that's 10,000 separate count operations.

```sql
-- Works but slower on large data
SELECT name
FROM Employee e
WHERE (
    SELECT COUNT(*)
    FROM Employee r
    WHERE r.managerId = e.id
) >= 5;
```

---

**Efficiency ranking for this pattern:**

| Rank | Method | Why |
|---|---|---|
| 1 | Self JOIN + GROUP BY + HAVING | Single query, no subquery, elegant |
| 2 | Aggregate in CTE/subquery + JOIN back | Equally efficient, cleaner separation of logic |
| 3 | Correlated subquery | Runs count per row — slower on large tables |

---

**WHERE vs HAVING — When to Use Which**

| Clause | Filters on | Runs when | Example |
|---|---|---|---|
| `WHERE` | Individual rows (before grouping) | Before `GROUP BY` | `WHERE status = 'active'` |
| `HAVING` | Aggregated groups (after grouping) | After `GROUP BY` | `HAVING COUNT(*) > 3` |

**In plain language:** Think of it as two gates. The first gate (WHERE) decides which individual rows are allowed into the grouping party. The second gate (HAVING) looks at the groups that formed and decides which groups are allowed into the final result. You can use both in the same query — WHERE narrows the rows first, then GROUP BY collapses them, then HAVING narrows the groups.

```sql
-- "Active customers who placed more than 5 orders in 2024"
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM Orders
WHERE status = 'completed'                        -- gate 1: only completed orders
  AND EXTRACT(YEAR FROM order_date) = 2024        -- gate 1: only 2024
GROUP BY customer_id
HAVING COUNT(*) > 5;                              -- gate 2: only groups with 5+
```

<hr style="border: 2px solid black;">

<a id='pattern-k-pivoting'></a>

### K. Pivoting (Rows to Columns)

**Signals:** show each category as a column, side by side, pivot, crosstab, transpose

---

**Pivoting with CASE + GROUP BY — Turning Rows into Columns**

**Method:** Combine `CASE` expressions inside aggregate functions with `GROUP BY` to transform row values into separate columns. This is the most portable approach — it works in every SQL dialect. Some dialects also have a dedicated `PIVOT` keyword, but the CASE approach is what interviewers typically expect.

**In plain language:** Imagine you have a table where each row is a student's score for a different subject (one row for Math, one for Science, one for English). The interviewer wants you to show one row per student with Math, Science, and English as separate columns. You use CASE to say "if the subject is Math, grab the score" and wrap it in MAX or SUM so each CASE produces one value per student.

```sql
SELECT
    student_id,
    MAX(CASE WHEN subject = 'Math' THEN score END) AS math_score,
    MAX(CASE WHEN subject = 'Science' THEN score END) AS science_score,
    MAX(CASE WHEN subject = 'English' THEN score END) AS english_score
FROM Scores
GROUP BY student_id;
```

---

**Why MAX() around the CASE?**

**In plain language:** After GROUP BY collapses the rows, each group might have multiple rows (one per subject). The CASE picks the score only when the subject matches and returns NULL for the rest. MAX() grabs that one non-NULL value from the group. You could also use MIN() or SUM() — it doesn't matter when there's only one non-NULL value per group. The point is you need *some* aggregate function to satisfy GROUP BY.

---

**COUNT(DISTINCT ...) — Counting Unique Values in Groups**

**Signals:** how many different, number of unique, distinct count per group

**Method:** `COUNT(DISTINCT column)` inside a GROUP BY counts only the unique values of that column within each group. Regular `COUNT(*)` counts all rows; `COUNT(DISTINCT ...)` deduplicates before counting.

**In plain language:** If a customer bought the same product 3 times, `COUNT(*)` would say 3 orders but `COUNT(DISTINCT product_id)` would say 1 unique product. This comes up when the question asks "how many *different* products" rather than "how many orders."

```sql
SELECT
    customer_id,
    COUNT(*) AS total_orders,
    COUNT(DISTINCT product_id) AS unique_products
FROM Orders
GROUP BY customer_id;
```

<a id='pattern-l-string-text-operations'></a>

### L. String & Text Operations

**Signals:** like, contains, starts with, email, valid, case, uppercase, lowercase, substring, concatenate, group strings, comma-separated list

---

**Pattern matching and text transformation — `LIKE`, regex (`~`), `LENGTH()`, and case/substring functions**

**Method:** Use `LIKE` with wildcards for simple pattern matching (SQL standard across dialects). For more complex patterns, use regex operators like `~` (PostgreSQL). For text transformations, use `UPPER()`, `LOWER()`, `LEFT()`, `SUBSTRING()`, `CONCAT()`, or `||` (concatenation operator). For aggregating strings per group, use `STRING_AGG()` or `GROUP_CONCAT()`.

**In plain language:** String operations let you filter based on text patterns, manipulate text (change case, extract parts, join pieces), and group text values together. These are often used to clean or validate data — for example, finding patients with conditions starting with a certain code, fixing name capitalization, or listing all products sold in each category.

```sql
-- Example 1: Find patients with condition starting with 'DIAB'
SELECT patient_id, condition_code
FROM Patients
WHERE condition_code LIKE 'DIAB%';

-- Example 2: Validate email format (PostgreSQL regex)
SELECT email
FROM Users
WHERE email ~ '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}$';

-- Example 3: Find tweets longer than 15 characters
SELECT tweet_id
FROM Tweets
WHERE LENGTH(text) > 15;

-- Example 4: Fix name capitalization
SELECT
    user_id,
    CONCAT(UPPER(LEFT(name, 1)), LOWER(SUBSTRING(name, 2))) AS fixed_name
FROM Users;

-- Example 5: List products sold per category (concatenate with commas)
SELECT
    category,
    STRING_AGG(product_name, ', ') AS products_list
FROM Products
GROUP BY category;
```

---

**Common String Functions**

| Function | Purpose | Example | Dialect Notes |
|---|---|---|---|
| `LIKE 'pattern%'` | Pattern matching (% = any chars, _ = single char) | `name LIKE 'J%'` (starts with J) | Universal |
| `~` (regex operator) | Regular expression matching | `email ~ '^[A-Za-z0-9]+@' ` | PostgreSQL |
| `LENGTH()` / `LEN()` | Character count | `LENGTH(name) > 5` | MySQL/PostgreSQL: LENGTH; SQL Server: LEN |
| `UPPER()` / `LOWER()` | Case conversion | `UPPER(status)` | Universal |
| `LEFT(str, n)` | First n characters | `LEFT(code, 3)` | Most dialects; substring-based in some |
| `SUBSTRING(str, start, len)` | Extract substring | `SUBSTRING(phone, 1, 3)` | Universal |
| `CONCAT(str1, str2, ...)` / `||` | Concatenate strings | `CONCAT(first_name, ' ', last_name)` or `first_name \|\| ' ' \|\| last_name` | CONCAT universal; \|\| in PostgreSQL/Oracle |
| `STRING_AGG(str, sep)` / `GROUP_CONCAT()` | Aggregate strings per group | `STRING_AGG(name, ', ')` | PostgreSQL: STRING_AGG; MySQL: GROUP_CONCAT |
| `TRIM()` / `LTRIM()` / `RTRIM()` | Remove whitespace | `TRIM(name)` | Universal |

---

**How to recognize this pattern:**

The question mentions **text matching**, **text extraction**, **case conversion**, or **combining text values**. Examples:
- "Find emails that don't match the format..."
- "Show patient conditions starting with..."
- "List all products sold in each category as a comma-separated string..."
- "Fix names by capitalizing the first letter..."
- "Find invalid tweets (longer than N characters)..."

---

> **Interview tip:** String operations vary significantly by dialect — PostgreSQL uses `~` for regex, MySQL uses `REGEXP`, SQL Server has different function names. Always ask which dialect you're using or state your assumptions. When building regex patterns, start simple (`LIKE`) and escalate to regex only if the pattern is too complex for LIKE wildcards. Aggregating strings with `STRING_AGG` / `GROUP_CONCAT` is an intermediate-level trick — mention it to show you know how to clean up output.

<a id='pattern-m-conditional-aggregation'></a>

### M. Conditional Aggregation (CASE inside SUM/COUNT/AVG)

**Signals:** rate, percentage, approval rate, confirmation rate, approved vs total, monthly breakdown by category, ratio, proportion, approved amount alongside total amount

---

**Computing rates and conditional sums — `CASE` inside `COUNT()`, `SUM()`, or `AVG()`**

**Method:** Wrap `CASE WHEN` inside an aggregate function like `SUM()`, `COUNT()`, or `AVG()` to compute derived values per group. For rates, use `AVG(CASE WHEN condition THEN 1.0 ELSE 0 END)` to get a proportion. For separate counts/sums of different categories, use multiple `CASE` expressions in the same query. When the logic requires identifying a "first" or "special" row first, use a subquery with window functions (like `ROW_NUMBER()`), then apply conditional aggregation on the result.

**In plain language:** Instead of running multiple queries (one for "approved" transactions, one for "all" transactions), you use `CASE` to label each row inside the aggregation. Then `SUM()`, `COUNT()`, or `AVG()` processes only the rows that match. This is much faster than filtering with WHERE and re-grouping.

```sql
-- Example 1: Approval rate per user
SELECT
    user_id,
    COUNT(*) AS total_requests,
    SUM(CASE WHEN status = 'approved' THEN 1 ELSE 0 END) AS approved_count,
    AVG(CASE WHEN status = 'approved' THEN 1.0 ELSE 0 END) AS approval_rate
FROM Requests
GROUP BY user_id;

-- Example 2: Monthly transaction breakdown (approved vs total amount)
SELECT
    TO_CHAR(created_at, 'YYYY-MM') AS month,
    country,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount,
    SUM(CASE WHEN status = 'approved' THEN amount ELSE 0 END) AS approved_amount,
    ROUND(100.0 * SUM(CASE WHEN status = 'approved' THEN amount ELSE 0 END) / SUM(amount), 2) AS approval_percentage
FROM Transactions
GROUP BY TO_CHAR(created_at, 'YYYY-MM'), country
ORDER BY month, country;

-- Example 3: Confirmation rate (first order per customer only)
-- Step 1: Identify first order per customer
-- Step 2: Calculate confirmation rate on first orders
WITH first_orders AS (
    SELECT
        customer_id,
        order_date,
        preferred_delivery_date,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_num
    FROM Orders
)
SELECT
    customer_id,
    COUNT(*) AS first_order_count,
    SUM(CASE WHEN order_date = preferred_delivery_date THEN 1 ELSE 0 END) AS confirmed_on_time,
    AVG(CASE WHEN order_date = preferred_delivery_date THEN 1.0 ELSE 0 END) AS confirmation_rate
FROM first_orders
WHERE order_num = 1
GROUP BY customer_id;

-- Example 4: Quality percentage (average rating scaled by position)
SELECT
    query_id,
    ROUND(AVG(rating / NULLIF(position, 0)), 4) AS quality_rating
FROM QueryRatings
GROUP BY query_id;
```

---

**Key CASE Patterns for Aggregation**

| Pattern | Use Case | Example |
|---|---|---|
| `SUM(CASE WHEN cond THEN 1 ELSE 0 END)` | Count rows matching condition | `SUM(CASE WHEN status = 'approved' THEN 1 ELSE 0 END)` |
| `COUNT(CASE WHEN cond THEN 1 END)` | Same as above (COUNT ignores NULLs) | `COUNT(CASE WHEN status = 'approved' THEN 1 END)` |
| `AVG(CASE WHEN cond THEN 1.0 ELSE 0 END)` | Compute rate/percentage | `AVG(CASE WHEN confirmed THEN 1.0 ELSE 0 END)` → 0.75 = 75% |
| `SUM(CASE WHEN cond THEN amount ELSE 0 END)` | Sum values matching condition | `SUM(CASE WHEN state = 'approved' THEN amount ELSE 0 END)` |
| Multiple CASE per GROUP BY | Different aggregations in one pass | `SUM(amount)`, `SUM(CASE ... approved ...)`, `SUM(CASE ... rejected ...)` all in one query |
| Subquery + CASE | Identify special rows first (e.g., "first per customer"), then aggregate | CTE with ROW_NUMBER, then filter WHERE rn = 1, then apply conditional aggregation |

---

**How to recognize this pattern:**

The question asks for **rates**, **percentages**, **counts of matching rows**, or **breakdowns by category**. Examples:
- "What percentage of orders were confirmed?"
- "Show approved and total transaction amounts per month."
- "Calculate the approval rate per user."
- "Find the confirmation rate for each customer's first order only."
- "Show monthly transaction counts and average approved amounts."

Key signal: The output includes both **the raw count/sum** and a **derived rate/percentage**, OR you need different sums for different subsets of rows (approved vs rejected, confirmed vs not confirmed).

---

> **Interview tip:** Conditional aggregation is a **mid-level SQL skill**. Interviewers love seeing it because it shows you understand how to compute multiple metrics in a single GROUP BY — which is much faster than running separate queries. When you spot a "rate" or "percentage" in the problem, think **CASE inside COUNT/SUM/AVG immediately**. Also watch for the **two-step subquery pattern**: first, use a window function to mark special rows (like "first order per customer"), then apply conditional aggregation on the subquery result. This combo handles tricky questions like "confirmation rate on first orders only."

<hr style="border: 3px solid black;">

<a id='4-critical-efficiency-rules'></a>

## 4. Critical Efficiency Rules

### Rule 1 — Avoid Duplicates

| Bad | Good |
|---|---|
| Window + `DISTINCT` | `GROUP BY` |

Using `DISTINCT` after a window function means the window computed values for rows that will be thrown away. Use `GROUP BY` to aggregate upfront.

### Rule 2 — Avoid Repeated Work

| Bad | Good |
|---|---|
| Correlated subquery | Window / Join |

Correlated subqueries re-execute for every row in the outer query. Window functions and joins compute results in a single pass.

### Rule 3 — Match Tool to Structure

| Need | Use |
|---|---|
| Previous row | `LAG` |
| One row per group | `GROUP BY` |
| Keep all rows | Window function |
| Pair rows | Join |

<hr style="border: 3px solid black;">

<a id='5-group-by-vs-window-functions'></a>

## 5. GROUP BY vs Window Functions

This is one of the **most important concepts** to understand for SQL interviews.

| Feature | GROUP BY | Window Function |
|---|---|---|
| **Effect on rows** | Collapses (reduces) rows | Keeps all rows |
| **Output** | 1 row per group | All original rows + new column |
| **Use when** | You need aggregated results only | You need row-level + group-level data |

### Example: GROUP BY Result

| machine | avg_time |
|---|---|
| A | 3.5 |
| B | 2.1 |

### Example: Window Function Result

| machine | process | time | avg_time |
|---|---|---|---|
| A | 1 | 3.0 | 3.5 |
| A | 2 | 4.0 | 3.5 |
| B | 1 | 2.1 | 2.1 |

### Memory Rule

| Rule | Meaning |
|---|---|
| Look across rows → **Window** | Comparisons |
| Collapse rows → **GROUP BY** | Aggregation |
| Pair rows → **Join** | Start/end matching |
| Keep rows + add info → **Window** | Group context |
| Need anti-match → **NOT EXISTS** | Missing data |

<hr style="border: 3px solid black;">

<a id='6-method-selection-framework'></a>

## 6. Method Selection: How to Know All Your Options

Once you identify the pattern, there are almost always **multiple valid SQL approaches**. The key skill isn't just picking the right pattern — it's knowing all the methods available for that pattern and choosing the most efficient one.

### The 3-Method Mental Model

For nearly every single-table pattern, your options fall into three categories:

| # | Method Type | How It Works | Typical Efficiency |
|---|---|---|---|
| 1 | **Direct aggregation** (`GROUP BY` + aggregate functions) | Collapse rows in one pass | Best — single scan |
| 2 | **Window function** (`OVER()`) | Compute across rows without collapsing | Great — single scan, keeps all rows |
| 3 | **Correlated subquery** | Re-run a subquery for every row | Weakest — runs N times for N rows |

### How to Enumerate Your Options — Decision Flow

Use this flow after you've identified the pattern. It tells you which methods are available and which to pick:

<div class="fc">
  <div class="fc-node fc-start">I identified the pattern.<br/>Now: what are my method options?</div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">Does the output have FEWER rows than the input? (collapsing)</div>
  <div class="fc-node fc-good">NO → <strong>Use a WINDOW FUNCTION</strong><br/>(keeps all rows, adds computed column)<code>SELECT col, SUM(col) OVER(PARTITION BY grp) FROM t</code></div>
  <div class="fc-arrow fc-else">if yes (collapsing) ▼</div>

  <div class="fc-node fc-good">Start with <strong>GROUP BY</strong> (most efficient)<code>SELECT grp, COUNT(*) FROM t GROUP BY grp</code></div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">Does the output need columns that aren't in the GROUP BY?<br/>(e.g., names, details)</div>
  <div class="fc-node fc-good">NO → <strong>Done! GROUP BY + HAVING is your complete answer.</strong><code>SELECT grp, COUNT(*) FROM t
GROUP BY grp HAVING COUNT(*) > n</code></div>
  <div class="fc-arrow fc-else">if yes (need extra columns) ▼</div>

  <div class="fc-node fc-warn">Need to JOIN back to get extra columns. Pick your strategy:</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-three">
    <div class="fc-branch">
      <div class="fc-label fc-tag">Option A (Rank 1)</div>
      <div class="fc-node fc-good">Self JOIN + GROUP BY + HAVING<br/>Join first, then aggregate.<br/>One query, reads like English.<code>SELECT a.name, COUNT(b.id)
FROM t a JOIN t b ON a.grp = b.grp
GROUP BY a.name HAVING COUNT(*) > n</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">Option B (Rank 2)</div>
      <div class="fc-node fc-good">CTE/Subquery + JOIN back<br/>Aggregate first, then join.<br/>Clean separation of steps.<br/>Same performance as Option A.<code>WITH agg AS (
  SELECT grp, COUNT(*) AS cnt
  FROM t GROUP BY grp
)
SELECT t.name, agg.cnt
FROM t JOIN agg ON t.grp = agg.grp</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-warn">Option C (Rank 3 — avoid)</div>
      <div class="fc-node fc-warn">Correlated subquery<br/>Runs aggregate per row.<br/>Mention it, then reject it.<code>SELECT name,
  (SELECT COUNT(*) FROM t t2
   WHERE t2.grp = t1.grp) AS cnt
FROM t t1  -- runs N times!</code></div>
    </div>
  </div>
</div>

### CTE vs Subquery — They Are Wrappers, Not Strategies

A `WITH ... AS` (CTE) and an inline subquery do the **exact same work**. The CTE just gives it a name. PostgreSQL typically inlines CTEs, so performance is identical.

```sql
-- These two are equivalent in performance:

-- CTE version
WITH counts AS (
    SELECT department, COUNT(*) AS cnt
    FROM employees
    GROUP BY department
)
SELECT e.name, c.cnt
FROM employees e
JOIN counts c ON e.department = c.department;

-- Inline subquery version
SELECT e.name, c.cnt
FROM employees e
JOIN (
    SELECT department, COUNT(*) AS cnt
    FROM employees
    GROUP BY department
) c ON e.department = c.department;
```

**Rule of thumb:** Use a CTE when it makes the query easier to read. Use an inline subquery when the query is already short. Never choose based on performance — they're the same.


<hr style="border: 3px solid black;">

<a id='7-how-to-think-from-the-vignette'></a>

## 7. How to Think from the Vignette

Follow these steps **before writing any SQL**:

### Step 1 — Identify the Output Shape

> *What should the final table look like?*

Examples:
- Weather problem → list of IDs (one per qualifying day)
- Activity problem → 1 row per machine

### Step 2 — Ask: Row vs Group?

> *Am I comparing rows or combining rows?*

- Comparing → Window function territory
- Combining → GROUP BY territory

### Step 3 — Identify the Grouping Level

> *What is the unit of output?*

Examples:
- Per machine → `GROUP BY machine_id`
- Per customer → `GROUP BY customer_id`
- No grouping → no `GROUP BY`

### Step 4 — Identify Pairing Logic

> *Do I need to match rows together?*

Examples:
- Start + end → `CASE` aggregation or self join
- Today + yesterday → `LAG`

### Step 5 — Choose the Tool

Use the [Decision Tree](#2-decision-tree-5-second-version) and the [Master Pattern Table](#1-first-step-identify-the-pattern) to select the best approach.

---

### Worked Examples

**Weather Problem:**

| Step | Answer |
|---|---|
| Output shape | List of IDs |
| Row vs group | Comparing rows |
| Grouping level | None |
| Pairing logic | Today vs yesterday |
| **Tool** | **LAG** |

**Activity Problem:**

| Step | Answer |
|---|---|
| Output shape | 1 row per machine |
| Row vs group | Combining rows |
| Grouping level | machine_id, process_id |
| Pairing logic | Start + end timestamps |
| **Tool** | **GROUP BY + CASE** |

<hr style="border: 3px solid black;">

<a id='9-interview-prompt-templates'></a>

## 9. Interview Prompt Templates

Use these phrases during an interview to structure your thinking out loud:

| # | Template | When to Use |
|---|---|---|
| 1 | *"This is a [compare/combine/rank] problem because..."* | Pattern identification |
| 2 | *"The output is one row per [X], so I'll group by [X]."* | Output shape |
| 3 | *"I need to compare each row to the previous one, so I'll use LAG."* | Row comparison |
| 4 | *"I need to collapse rows into one, so I'll use GROUP BY."* | Aggregation |
| 5 | *"I need to match related rows, so I'll join or use CASE aggregation."* | Pairing rows |
| 6 | *"This avoids duplicate computation and reduces rows early."* | Efficiency reasoning |
| 7 | *"Window functions let me keep all rows while adding group-level values."* | Window explanation |
| 8 | *"I won't use DISTINCT because it removes duplicates after computation."* | Avoiding mistakes |
| 9 | *"Does this produce exactly one row per required output?"* | Final check |
| 10 | *"If window functions weren't available, I'd use a self join."* | Backup strategy |

<hr style="border: 3px solid black;">

<a id='subquery-decision-tree'></a>

## When Do I Need a Subquery?

Even in a **single-table** problem, you may need a subquery when the question asks you to compare individual rows or groups against a **global aggregate** that cannot be computed in the same pass.

### Subquery Decision Tree

<div class="fc">
  <div class="fc-node fc-start">Does my calculation need a value that comes from a DIFFERENT aggregation level?<br/>(e.g., "compare each row to the overall avg")</div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">Do I need a different aggregation level at all?</div>
  <div class="fc-node fc-good">NO → <strong>No subquery needed</strong> — use GROUP BY or window functions<code>SELECT grp, COUNT(*) FROM t GROUP BY grp
-- or --
SELECT col, SUM(col) OVER(PARTITION BY grp) FROM t</code></div>
  <div class="fc-arrow fc-else">if yes ▼</div>

  <div class="fc-node fc-start">Can I get the value with a window function instead?<br/>(e.g., AVG() OVER())</div>
  <div class="fc-node fc-good">YES → <strong>Use window function</strong> — cleaner and often faster<code>SELECT col,
  AVG(col) OVER() AS overall_avg
FROM t</code></div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-warn">YOU NEED A SUBQUERY — Where does it go?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-three">
    <div class="fc-branch">
      <div class="fc-label fc-tag">SELECT clause</div>
      <div class="fc-node fc-action">Show a global value alongside each row<code>SELECT grp, COUNT(x) /
  (SELECT COUNT(*) FROM t) * 100
  AS pct_of_total
FROM t GROUP BY grp</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">WHERE clause</div>
      <div class="fc-node fc-action">Filter rows based on a computed threshold<code>SELECT * FROM t
WHERE col > (SELECT AVG(col) FROM t)</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">FROM clause</div>
      <div class="fc-node fc-action">Pre-aggregate, then join or query the result<code>SELECT * FROM (
  SELECT grp, COUNT(*) AS cnt
  FROM t GROUP BY grp
) sub WHERE sub.cnt > 5</code></div>
    </div>
  </div>
</div>

---

### Quick Reference: Subquery Placement

| Put the subquery in… | When you need to… | Signal words |
|---|---|---|
| **SELECT** | Display a global value alongside grouped/row-level data | "percentage of total", "compared to overall", "ratio" |
| **WHERE** | Filter rows using a computed threshold | "above average", "greater than the median", "top performers" |
| **FROM** | Pre-aggregate data before applying further logic | "rank within groups", "filter aggregated results", "join summaries" |

> **Tip:** For single-table problems, SELECT and WHERE subqueries are most common. The subquery usually references the *same table* with a different aggregation level.


<hr style="border: 3px solid black;">

<a id='10-what-to-avoid'></a>

## 10. What to Avoid

| Bad Pattern | Why It's Bad | Better Alternative |
|---|---|---|
| Window + `DISTINCT` | Duplicate work — computes then discards | `GROUP BY` |
| Correlated subquery | Repeated work — runs per row | `LAG` / `JOIN` |
| `NOT IN` (with NULLs) | Incorrect results — NULLs break logic | `NOT EXISTS` |
| Self join for running totals | Heavy computation | `SUM() OVER()` |

<hr style="border: 3px solid black;">

<a id='11-final-takeaway'></a>

## 11. Final Takeaway

The entire game is:

```
Identify the shape  →  Pick the tool  →  Avoid unnecessary work
```

### Interview Script (Memorize This)

> *"I identify whether this is comparing rows or aggregating them. If it's row comparison, I use LAG. If it's aggregation, I use GROUP BY. If I need both row-level and group-level data, I use a window function."*